# From 13: Function Calling

This lesson covers how we can get agents (an LLM) to decide how to run the search.

In [10]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

from minsearch import Index

In [11]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)


index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

# fit an index on our documents like how in scikit-learn you fit a model on data
index.fit(documents)

#### Running without Tools

In [12]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Absolutely — in most cases, yes, you can join if the course is still open for enrollment.\n\nA few quick things to check:\n- **Enrollment deadline:** Has registration closed?\n- **Prerequisites:** Do you meet any required background or prior courses?\n- **Capacity:** Is the class full?\n- **Access method:** Some courses require instructor approval or a special registration link.\n\nIf you want, I can help you draft a short message to the instructor or check what to ask the registrar/course organizer.'

In [13]:


def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

## Creating a Tool

In [14]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

## Asking with a Tool

The following output will show a function_call entry. The LLM decided to call the search function.

Also the query was modiified.

In [15]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join late enrollment registration"}', call_id='call_4GjQ6tUb862AHvFYzxRfYXZT', name='search', type='function_call', id='fc_0d2d5b5b42cc2dc7006a3f7f5a2a98819d98eaac7fff9e7726', namespace=None, status='completed')]

In [16]:
import json

# Parse the JSON arguments for the function call
call = response.output[0]

In [17]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join late enrollment registration"}', call_id='call_4GjQ6tUb862AHvFYzxRfYXZT', name='search', type='function_call', id='fc_0d2d5b5b42cc2dc7006a3f7f5a2a98819d98eaac7fff9e7726', namespace=None, status='completed')

We take the JSON arguments given to us by the LLM and parse them.

In [18]:
args = json.loads(call.arguments)
args

{'query': 'join course discovered can I join late enrollment registration'}

We call with search functon, and serialize the result (put to JSON again).

In [19]:

results = search(**args)
result_json = json.dumps(results, indent=2)
result_json

'[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?",\n    "answer": "You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the

In [20]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

# CALLING THE MODEL AGAIN WITH EXPANDED MESSAGE
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf your goal is to get a certificate, make sure you submit your project while submissions are still open. If you just want to learn, you can start anytime.'

Now we are calling the LLM 2x. 

Agentic RAG aka tool use aka function calling means the LLM will 
1. Decide what tools to call and send back results.
2. Decide how to process the results and give answer.

## Cost

In [21]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(770, 46)

In [22]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


Since we call the API twice, we calculated the usage of the 2nd API call. 

More calls = more cost.

# Agent Anatomy

An agent keeps calling the model and running tools until it's done. 

1. Instructions
2. Tools (functions agent can call)
3. Memory (complete message history, appended every prompt)

### Instructions / Developer Prompt

In [23]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [24]:
# function call helper, so we can call it multiple times
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [25]:
question = "I just discovered the course. Can I join it?"

# initial message history has just the developer prompt (system instructions)
# and user question
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

# get response
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

# for any function call needed, make_call
for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course enroll late discovered the course can I join"}
function_call: search {"query":"course registration late enrollment join after course started FAQ"}
function_call: search {"query":"I just discovered the course can I join"}


### Full Agent Loop

In [ ]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join the course late discovered course can I join enrollment late join course FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course even if you just discovered it.

A couple of notes:
- You can start anytime and work through the materials at your own pace.
- If you want a certificate, you need to complete the course with the live cohort and submit your project while submissions are still open.

If you want, I can also help you find the best way to get started or explain the certificate rules in more detail.


### Wrapping it in a Function

In [29]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [30]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run Ollama local install run locally course FAQ"}
function_call: search {"query":"Ollama local setup run local model FAQ"}
function_call: search {"query":"Olama local machine installation command FAQ"}
iteration #2...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - macOS: download the `.pkg` from https://ollama.com/download
   - Windows: download the `.msi` from the same page
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This downloads the model and starts it locally.

3. **Check the local server**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from the Ollama server.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content":

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - macOS: download the `.pkg` from https://ollama.com/download\n   - Windows: download the `.msi` from the same page\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and starts it locally.\n\n3. **Check the local server**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from the Ollama server.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection error, restart the server with:\n```bash\nollama serve\n```\nor in a notebook:\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```\n\nIf you want, I can

In [31]:
# Let's push the model to explore more with more searches
# by re-writing the instructions

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [32]:
agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment FAQ"}
iteration #2...
function_call: search {"query":"certificate submit project while accepting submissions live cohort self-paced join course registration interest FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open, and the certificate is only available if you complete it with a live cohort.

If you'd like, I can also help you with:
- how to start the course,
- whether you can do it self-paced,
- or how the certificate/peer-review process works.

Is there another area you want to explore?


"Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open, and the certificate is only available if you complete it with a live cohort.\n\nIf you'd like, I can also help you with:\n- how to start the course,\n- whether you can do it self-paced,\n- or how the certificate/peer-review process works.\n\nIs there another area you want to explore?"

### Restricting Off-Topic Questions

In [33]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit what is queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit opening chess definition decline accepted move d4 d5 c4"}
iteration #3...
ASSISTANT:
A Queen’s Gambit is a chess opening that starts with:

1. **d4 d5**
2. **c4**

White offers the **c-pawn** to try to **draw Black’s d-pawn away from the center** and gain better control of the center.

There are two main responses:
- **Queen’s Gambit Accepted**: Black takes the pawn with `dxc4`
- **Queen’s Gambit Declined**: Black does not take it and instead supports the center

So, in simple terms: it’s a classic opening where White sacrifices a pawn temporarily to get a stronger position.

If you want, I can also explain:
- the ideas behind it,
- the difference between accepted and declined,
- or show a few common move lines.


'A Queen’s Gambit is a chess opening that starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** to try to **draw Black’s d-pawn away from the center** and gain better control of the center.\n\nThere are two main responses:\n- **Queen’s Gambit Accepted**: Black takes the pawn with `dxc4`\n- **Queen’s Gambit Declined**: Black does not take it and instead supports the center\n\nSo, in simple terms: it’s a classic opening where White sacrifices a pawn temporarily to get a stronger position.\n\nIf you want, I can also explain:\n- the ideas behind it,\n- the difference between accepted and declined,\n- or show a few common move lines.'

In [ ]:
# changing instructions again to only answer FAQ

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()








agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening course FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find a course-related FAQ entry for “queen gambit,” so it looks like this may be off-topic or not covered in the course FAQ.

If you meant **Queen’s Gambit** in chess, I can’t explain it from the FAQ database. If you meant something else course-related, please rephrase and I’ll look it up.

Are there other areas you want to explore?


'I couldn’t find a course-related FAQ entry for “queen gambit,” so it looks like this may be off-topic or not covered in the course FAQ.\n\nIf you meant **Queen’s Gambit** in chess, I can’t explain it from the FAQ database. If you meant something else course-related, please rephrase and I’ll look it up.\n\nAre there other areas you want to explore?'

Note: A real guardrail will check the input before the agent even runs. (This is a separate topic.)

Every agent framework uses this same pattern / agentic loop.

# ToyAI Kit

In [35]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [37]:
# Register the tool (search function)

agent_tools = Tools()
agent_tools.add_tool(search, search_tool)


In [38]:
search_tool

{'type': 'function',
 'name': 'search',
 'description': 'Search the FAQ database for entries matching the given query.',
 'parameters': {'type': 'object',
  'properties': {'query': {'type': 'string',
    'description': 'Search query text to look up in the course FAQ.'}},
  'required': ['query'],
  'additionalProperties': False}}

#### Let ToyAIKit generate a schema for the function

In [ ]:
# if this is our function
# add a doc string
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [40]:
# register without passing a schema
agent_tools = Tools()
agent_tools.add_tool(search)

In [41]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

The output is generated from the doc string and type hint.

OpenAI Agents SDK, PydanticAI, LangChain, Google ADK all work as such.

### Adding a Chat Interface and Runner (Agent)

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

# an agent that runs the agent loop,
# sends messages, executes function calls,
# adds tool outputs back
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [43]:
# Try running 1 prompt with Olama typo
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)


-> Response received


-> Response received


-> Response received


#### Cost and tokens

In [ ]:
# ToyAi Kit Framework keeps a running cost
result.cost

CostInfo(input_cost=Decimal('0.00268875'), output_cost=Decimal('0.001503'), total_cost=Decimal('0.00419175'))

In [45]:
# Framework also provides full message history
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run Ollama local install course FAQ"}', call_id=

### Continuing Conversation via Message History

In [46]:
# BECAUSE WE EXTENDED/APPENDED THE MESSAGES
# the runner agent know we're referring to Ollama with different model
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [ ]:
# Interactive Chat version
# type "stop" to exit
runner.run()

-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='When does the course for 2026 start?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"2026 course start date when doe